# DRUGseqPy — Notebook 3: Dose-Response & Cheminformatics

1. 4-parameter log-logistic (LL4) dose-response fitting with lmfit
2. Model selection: LL4 vs Weibull families (AIC)
3. EC50 confidence intervals
4. Multi-compound parallelised fitting
5. Dose-response panel plots
6. SMILES retrieval from PubChem API
7. Molecular descriptor computation (rdkit)
8. Structure–activity relationship (SAR) modelling

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp

import drugseqpy as ds
from drugseqpy.utils import make_dummy_screen
from drugseqpy.dose_response import fit_dose_response, compute_multi_dr, plot_dr_panel

plt.rcParams['figure.dpi'] = 100
print(f'drugseqpy v{ds.__version__}')

## 0. Build a dose-response experiment

In [ ]:
# Build a dataset with multiple dose points
rng = np.random.default_rng(42)

doses = [0, 0.01, 0.1, 1, 10, 100]   # µM
n_reps = 4
n_genes = 200
compounds = ['CompA', 'CompB', 'CompC']

# Metadata
obs_rows = []
for cmpd in compounds:
    for dose in doses:
        for r in range(n_reps):
            obs_rows.append({
                'compound': cmpd if dose > 0 else 'DMSO',
                'dose': dose, 'dose_unit': 'uM',
                'plate_id': 'Plate01',
                'well_id': f'A{len(obs_rows)+1:02d}',
                'sample_type': 'DMSO' if dose == 0 else 'treatment',
            })
obs = pd.DataFrame(obs_rows)
obs.index = [f'S{i:04d}' for i in range(len(obs))]

# Counts: LL4 dose-response signal in first 20 genes per compound
n_samp = len(obs)
mu_base = rng.lognormal(5, 1.5, n_genes)
mat = rng.negative_binomial(n=5, p=5/(5+mu_base), size=(n_samp, n_genes)).astype(float)

for ci, cmpd in enumerate(compounds):
    ec50 = 0.5 * (10 ** ci)   # EC50 increases per compound
    g_start = ci * 20
    g_end   = g_start + 20
    for i, row in obs.iterrows():
        if row['compound'] == cmpd:
            d = row['dose']
            fold = 1 + 4 / (1 + (ec50 / (d + 1e-9)) ** 1.5)   # Hill equation
            idx = obs.index.get_loc(i)
            mat[idx, g_start:g_end] *= fold

counts = pd.DataFrame(mat.T.astype(int),
                       index=[f'Gene{i:04d}' for i in range(n_genes)],
                       columns=obs.index)
print(f'Dose-response dataset: {counts.shape[0]} genes × {counts.shape[1]} samples')
obs.groupby(['compound','dose']).size().unstack(fill_value=0)

In [ ]:
dsd_dr = ds.create_drugseq_object(counts=counts, obs=obs)
ds.normalize_counts(dsd_dr, method='limma_voom', inplace=True)

# Run DE at the top dose (collapsed) for gene ranking
ds.compute_multi_de(
    dsd_dr,
    reference='DMSO',
    method='ols_voom',
    within_plate=False,
    inplace=True,
)

## 1. Single-compound dose-response fitting

Uses **lmfit** (Python equivalent of R's drc package) with the 4-parameter
log-logistic model (Hill equation). EC50 confidence intervals are computed
via lmfit's built-in uncertainty propagation.

In [ ]:
dr_compA = fit_dose_response(
    dsd_dr,
    compound='CompA',
    compound_col='compound',
    dose_col='dose',
    reference='DMSO',
    n_genes_fit=50,
    model_selection=False,   # LL4 only (faster)
    min_r2=0.3,
)

print('\nTop 10 genes by R²:')
print(dr_compA.nlargest(10, 'r_squared')[
    ['gene','model','EC50','EC50_ci_lower','EC50_ci_upper','slope','r_squared','converged']
].to_string(index=False))

## 2. AIC-based model selection (LL4 / Weibull)

In [ ]:
dr_compA_sel = fit_dose_response(
    dsd_dr,
    compound='CompA',
    n_genes_fit=20,
    model_selection=True,   # compare LL4, W14, W24
    min_r2=0.3,
)

print('Model selection results:')
print(
    dr_compA_sel[dr_compA_sel['converged']]
    [['gene','model','EC50','r_squared','aic']]
    .sort_values('aic')
    .to_string(index=False)
)

# Model frequency
print('\nBest model counts:')
print(dr_compA_sel['model'].value_counts())

## 3. Dose-response panel plot

In [ ]:
fig = plot_dr_panel(
    dr_compA,
    dsd_dr,
    compound='CompA',
    n_genes=9,
    ncol=3,
)
plt.show()

## 4. Multi-compound dose-response

In [ ]:
dr_all = compute_multi_dr(
    dsd_dr,
    compounds=['CompA', 'CompB', 'CompC'],
    n_jobs=1,
    n_genes_fit=30,
    model_selection=False,
)

# Summary: converged genes and median EC50 per compound
for cmpd, df in dr_all.items():
    conv = df[df['converged']]
    print(f'{cmpd}: {len(conv)} converged fits, median EC50 = {conv["EC50"].median():.3f} µM')

In [ ]:
# EC50 comparison across compounds for shared responsive genes
shared_genes = set.intersection(*[set(df[df['converged']]['gene']) for df in dr_all.values()])
print(f'Genes converged in all compounds: {len(shared_genes)}')

if shared_genes:
    ec50_df = pd.DataFrame({
        cmpd: df.set_index('gene')['EC50']
        for cmpd, df in dr_all.items()
    }).loc[list(shared_genes)]

    fig, ax = plt.subplots(figsize=(6, 4))
    ec50_df.boxplot(ax=ax)
    ax.set_yscale('log')
    ax.set_ylabel('EC50 (µM)')
    ax.set_title('EC50 distribution across compounds')
    plt.tight_layout()
    plt.show()

## 5. SMILES retrieval from PubChem

Queries the PubChem REST API by compound name and caches results in `adata.uns`.
Requires an internet connection.

In [ ]:
import requests

def fetch_smiles(name: str) -> str | None:
    """Fetch canonical SMILES from PubChem by compound name."""
    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/property/CanonicalSMILES/JSON'
    try:
        r = requests.get(url, timeout=8)
        if r.ok:
            return r.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
    except Exception:
        pass
    return None

# Example with real drug names
real_drugs = ['imatinib', 'erlotinib', 'gefitinib']
smiles_map = {}
for drug in real_drugs:
    smi = fetch_smiles(drug)
    smiles_map[drug] = smi
    print(f'{drug}: {smi}')

## 6. Molecular descriptors with RDKit

Computes 2D physicochemical descriptors (MW, LogP, TPSA, HBD, HBA, RingCount)
from SMILES strings.

In [ ]:
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors

    def compute_descriptors(smiles_dict: dict) -> pd.DataFrame:
        rows = []
        for name, smi in smiles_dict.items():
            if smi is None:
                continue
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue
            rows.append({
                'compound':  name,
                'MW':        Descriptors.MolWt(mol),
                'LogP':      Descriptors.MolLogP(mol),
                'TPSA':      rdMolDescriptors.CalcTPSA(mol),
                'HBD':       rdMolDescriptors.CalcNumHBD(mol),
                'HBA':       rdMolDescriptors.CalcNumHBA(mol),
                'RingCount': rdMolDescriptors.CalcNumRings(mol),
                'RotBonds':  rdMolDescriptors.CalcNumRotatableBonds(mol),
            })
        return pd.DataFrame(rows)

    desc_df = compute_descriptors(smiles_map)
    print(desc_df.to_string(index=False))

except ImportError:
    print('RDKit not installed. Install: pip install rdkit\n'
          'Or via conda: conda install -c conda-forge rdkit')

## 7. Structure–Activity Relationship (SAR) modelling

Trains a Random Forest (or other sklearn estimator) to predict the number
of significant DE genes from molecular descriptors.
This extends the macpie SAR vignette by adding cross-validated model
comparison via sklearn pipelines.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

def train_sar_model(
    descriptors: pd.DataFrame,   # rows = compounds, numeric feature columns
    targets: pd.Series,           # one value per compound (e.g., n_sig_DE)
    methods: list = None,
    cv: int = 3,
) -> pd.DataFrame:
    """
    Cross-validated SAR modelling with multiple estimators.
    Returns a DataFrame with R² per method.
    """
    if methods is None:
        methods = {
            'Random Forest':    RandomForestRegressor(n_estimators=200, random_state=42),
            'Gradient Boost':   GradientBoostingRegressor(n_estimators=200, random_state=42),
            'Ridge Regression': Ridge(alpha=1.0),
        }

    feat_cols = descriptors.select_dtypes(include=np.number).columns.tolist()
    X = descriptors[feat_cols].values
    y = targets.values

    kf = KFold(n_splits=cv, shuffle=True, random_state=42)
    rows = []
    for name, estimator in methods.items():
        pipe = Pipeline([('scaler', StandardScaler()), ('model', estimator)])
        scores = cross_val_score(pipe, X, y, cv=kf, scoring='r2')
        rows.append({'method': name, 'mean_r2': scores.mean(),
                     'std_r2': scores.std(), 'cv_scores': scores.tolist()})
        print(f'  {name:<22}: R² = {scores.mean():.3f} ± {scores.std():.3f}')

    return pd.DataFrame(rows)


# Demonstration with synthetic descriptors (replace with real rdkit descriptors)
rng2 = np.random.default_rng(7)
n_demo = 30
demo_desc = pd.DataFrame({
    'MW':        rng2.uniform(200, 600, n_demo),
    'LogP':      rng2.uniform(0, 5, n_demo),
    'TPSA':      rng2.uniform(30, 150, n_demo),
    'HBD':       rng2.integers(0, 5, n_demo).astype(float),
    'HBA':       rng2.integers(0, 10, n_demo).astype(float),
    'RingCount': rng2.integers(1, 6, n_demo).astype(float),
})
# Synthetic activity: correlated with LogP (replace with real n_sig_DE values)
demo_activity = pd.Series(5 * demo_desc['LogP'] + rng2.normal(0, 2, n_demo),
                            name='n_sig_DE')

print('SAR cross-validation (R²):')
sar_results = train_sar_model(demo_desc, demo_activity, cv=3)
print(sar_results[['method','mean_r2','std_r2']].to_string(index=False))

In [ ]:
# Feature importance from best model (Random Forest)
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

feat_cols = demo_desc.select_dtypes(include=np.number).columns.tolist()
X = StandardScaler().fit_transform(demo_desc[feat_cols])
y = demo_activity.values

rf = RandomForestRegressor(n_estimators=500, random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=feat_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(5, 3))
importances.plot.barh(ax=ax, color='#2980B9', edgecolor='none')
ax.set_title('Random Forest feature importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 8. Export dose-response results

In [ ]:
for cmpd, df in dr_all.items():
    df.to_csv(f'dr_{cmpd}.csv', index=False)
    print(f'Saved dr_{cmpd}.csv ({len(df)} genes, {df["converged"].sum()} converged)')

---
**End of DRUGseqPy notebook series.**

For questions or contributions, open an issue on the project repository.